# 🛡️ InsurGig AI — Risk Prediction Model Training

This notebook trains a **Random Forest Regressor** to predict gig worker disruption risk scores based on environmental and traffic parameters.

### What This Does:
1. **Generates a synthetic dataset** of 10,000 gig worker disruption scenarios
2. **Saves the dataset** as `insurgig_dataset.csv` for reference
3. **Trains a Random Forest model** using README-defined weight formula
4. **Evaluates accuracy** with MSE and R² metrics
5. **Exports `risk_model.pkl`** ready to drop into the FastAPI backend

### Risk Score Formula (from README):
```
Risk Score = 0.4 × Weather + 0.3 × AQI + 0.2 × Traffic + 0.1 × Demand Drop
```

---

## Step 1: Install Dependencies

In [ ]:
!pip install scikit-learn pandas numpy matplotlib seaborn -q
print('✅ All dependencies installed!')

## Step 2: Generate Synthetic Dataset

Since we don't have real historical data, we generate realistic synthetic data based on Indian weather patterns, AQI ranges, and traffic conditions typical for gig delivery workers.

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)
n_samples = 10000

# --- Feature Generation (Indian City Ranges) ---
# Rainfall: 0-200mm (monsoon peaks)
rainfall = np.random.exponential(scale=25, size=n_samples)
rainfall = np.clip(rainfall, 0, 200)

# Temperature: 18-48°C (Indian climate range)
temperature = np.random.normal(loc=33, scale=6, size=n_samples)
temperature = np.clip(temperature, 18, 48)

# AQI: 30-500 (good to hazardous)
aqi = np.random.lognormal(mean=4.5, sigma=0.6, size=n_samples)
aqi = np.clip(aqi, 30, 500).astype(int)

# Traffic Index: 1-10 scale
traffic_index = np.random.uniform(1, 10, n_samples)

# Demand Drop %: 0-100
demand_drop_pct = np.random.beta(2, 5, n_samples) * 100

# City and Zone (categorical features)
cities = np.random.choice(['Hyderabad', 'Mumbai', 'Delhi', 'Bangalore', 'Chennai', 'Pune'], n_samples)
zones = np.random.choice(['Zone A', 'Zone B', 'Zone C'], n_samples)

print(f'✅ Generated {n_samples} synthetic samples')
print(f'   Rainfall range: {rainfall.min():.1f} - {rainfall.max():.1f} mm')
print(f'   Temperature range: {temperature.min():.1f} - {temperature.max():.1f} °C')
print(f'   AQI range: {aqi.min()} - {aqi.max()}')

## Step 3: Calculate Target Risk Scores

Using the weighted formula from the InsurGig README:
- **40%** Weather (rainfall or extreme heat)
- **30%** Air Quality (AQI)
- **20%** Traffic Congestion
- **10%** Demand Drop

In [ ]:
# Normalize features to 0-100 scale
norm_rain = np.clip((rainfall / 100) * 100, 0, 100)

# Weather score: use rain if raining, otherwise extreme heat
heat_score = np.clip(((temperature - 30) / 15) * 100, 0, 100)
weather_score = np.where(rainfall > 20, norm_rain, heat_score)

norm_aqi = np.clip((aqi / 400) * 100, 0, 100)
norm_traffic = (traffic_index / 10) * 100
norm_demand = demand_drop_pct

# Zone modifier
zone_modifier = np.where(zones == 'Zone A', 5, np.where(zones == 'Zone B', 10, 15))

# Core formula
risk_scores = (
    0.4 * weather_score +
    0.3 * norm_aqi +
    0.2 * norm_traffic +
    0.1 * norm_demand +
    zone_modifier
)

# Add realistic noise
risk_scores += np.random.normal(0, 3, n_samples)
risk_scores = np.clip(risk_scores, 0, 100)

print(f'✅ Risk scores calculated')
print(f'   Mean: {risk_scores.mean():.1f}')
print(f'   Std:  {risk_scores.std():.1f}')
print(f'   Min:  {risk_scores.min():.1f}, Max: {risk_scores.max():.1f}')

## Step 4: Build and Save Dataset

In [ ]:
data = pd.DataFrame({
    'city': cities,
    'zone': zones,
    'rainfall_mm': np.round(rainfall, 2),
    'temperature_c': np.round(temperature, 2),
    'aqi': aqi,
    'traffic_index': np.round(traffic_index, 2),
    'demand_drop_pct': np.round(demand_drop_pct, 2),
    'risk_score': np.round(risk_scores, 2)
})

# Derive risk level for reference
data['risk_level'] = data['risk_score'].apply(
    lambda x: 'High' if x > 75 else ('Medium' if x > 45 else 'Low')
)

# Save dataset
data.to_csv('insurgig_dataset.csv', index=False)
print(f'✅ Dataset saved as insurgig_dataset.csv ({len(data)} rows)')
print()
print('Risk Level Distribution:')
print(data['risk_level'].value_counts())
print()
data.head(10)

## Step 5: Visualize the Dataset

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('InsurGig AI — Synthetic Dataset Distribution', fontsize=16, fontweight='bold')

sns.histplot(data['rainfall_mm'], bins=50, ax=axes[0,0], color='#3b82f6', kde=True)
axes[0,0].set_title('Rainfall (mm)')

sns.histplot(data['temperature_c'], bins=50, ax=axes[0,1], color='#ef4444', kde=True)
axes[0,1].set_title('Temperature (°C)')

sns.histplot(data['aqi'], bins=50, ax=axes[0,2], color='#f59e0b', kde=True)
axes[0,2].set_title('AQI')

sns.histplot(data['traffic_index'], bins=50, ax=axes[1,0], color='#8b5cf6', kde=True)
axes[1,0].set_title('Traffic Index')

sns.histplot(data['risk_score'], bins=50, ax=axes[1,1], color='#021676', kde=True)
axes[1,1].set_title('Risk Score (Target)')

sns.countplot(x='risk_level', data=data, ax=axes[1,2], palette=['#22c55e', '#f59e0b', '#ef4444'], order=['Low', 'Medium', 'High'])
axes[1,2].set_title('Risk Level Distribution')

plt.tight_layout()
plt.savefig('dataset_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Visualization saved as dataset_distribution.png')

## Step 6: Train the Random Forest Model

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# Prepare features (numeric only for the model)
feature_cols = ['rainfall_mm', 'temperature_c', 'aqi', 'traffic_index', 'demand_drop_pct']
X = data[feature_cols]
y = data['risk_score']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Training set: {len(X_train)} samples')
print(f'Test set:     {len(X_test)} samples')
print()

# Train model
model = RandomForestRegressor(
    n_estimators=200,
    max_depth=12,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

print('🧠 Training Random Forest Regressor...')
model.fit(X_train, y_train)
print('✅ Model trained successfully!')

## Step 7: Evaluate Model Accuracy

In [ ]:
predictions = model.predict(X_test)

mse = mean_squared_error(y_test, predictions)
rmse = mse ** 0.5
mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print('📊 Model Accuracy Metrics:')
print(f'   R² Score:              {r2:.4f} (closer to 1.0 = better)')
print(f'   Mean Absolute Error:   {mae:.2f}')
print(f'   Root Mean Sq Error:    {rmse:.2f}')
print(f'   Mean Squared Error:    {mse:.2f}')
print()

# Feature importance
importance = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=False)

print('🔑 Feature Importance:')
for _, row in importance.iterrows():
    bar = '█' * int(row['Importance'] * 50)
    print(f'   {row["Feature"]:20s} {row["Importance"]:.4f} {bar}')

## Step 8: Visualize Model Performance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Predicted vs Actual
axes[0].scatter(y_test, predictions, alpha=0.3, s=10, c='#021676')
axes[0].plot([0, 100], [0, 100], 'r--', linewidth=2)
axes[0].set_xlabel('Actual Risk Score')
axes[0].set_ylabel('Predicted Risk Score')
axes[0].set_title(f'Predicted vs Actual (R² = {r2:.4f})')

# Feature Importance
axes[1].barh(importance['Feature'], importance['Importance'], color='#021676')
axes[1].set_xlabel('Importance')
axes[1].set_title('Feature Importance')

plt.tight_layout()
plt.savefig('model_performance.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Performance plots saved as model_performance.png')

## Step 9: Export Model for Backend

In [ ]:
import pickle

filename = 'risk_model.pkl'
with open(filename, 'wb') as file:
    pickle.dump(model, file)

print(f'🚀 SUCCESS: Model saved as "{filename}"!')
print(f'   File size: {__import__("os").path.getsize(filename) / 1024:.1f} KB')
print()
print('📋 Next Steps:')
print('   1. Download risk_model.pkl from the Files panel (left sidebar)')
print('   2. Place it in your backend/services/ directory')
print('   3. The FastAPI backend will automatically use it for predictions')

## Step 10: Quick Test — Predict Risk for a Sample Scenario

In [ ]:
# Test with a real-world scenario
test_scenarios = pd.DataFrame([
    {'rainfall_mm': 80, 'temperature_c': 28, 'aqi': 180, 'traffic_index': 8.0, 'demand_drop_pct': 45},
    {'rainfall_mm': 5, 'temperature_c': 42, 'aqi': 300, 'traffic_index': 3.0, 'demand_drop_pct': 10},
    {'rainfall_mm': 0, 'temperature_c': 30, 'aqi': 50, 'traffic_index': 2.0, 'demand_drop_pct': 5},
])

scenario_names = ['🌧️ Heavy Monsoon + Traffic', '🔥 Extreme Heat + Pollution', '☀️ Clear Day (Low Risk)']

preds = model.predict(test_scenarios)

print('🧪 Test Predictions:')
print('-' * 60)
for name, pred in zip(scenario_names, preds):
    level = 'High 🔴' if pred > 75 else ('Medium 🟡' if pred > 45 else 'Low 🟢')
    print(f'  {name}')
    print(f'  → Risk Score: {pred:.1f} | Level: {level}')
    print()

---
### ✅ Training Complete!

**Files Generated:**
- `insurgig_dataset.csv` — 10,000 row synthetic training dataset
- `risk_model.pkl` — Trained Random Forest model (ready for FastAPI)
- `dataset_distribution.png` — Data visualization
- `model_performance.png` — Model accuracy plots

Download `risk_model.pkl` and place it in your `backend/services/` directory.